In [1]:
import os
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
import os
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def extract_numeric_value(metric_str):
    """Extract mean value from strings like '0.5700 +- 0.0000' or '0.9428 ± 0.0000'"""
    try:
        # Handle both separator types
        if '+-' in metric_str:
            return float(metric_str.split('+-')[0].strip())
        elif '±' in metric_str:
            return float(metric_str.split('±')[0].strip())
        else:
            return float(metric_str)
    except:
        return None

def analyze_results(base_path='./logs'):
    data = []
    
    # Find all dataset folders
    dataset_folders = [f for f in os.listdir(base_path) if f.startswith('pca_SNN_DS1')]
    
    for dataset_folder in dataset_folders:
        dataset_path = os.path.join(base_path, dataset_folder)
        
        # Process each version folder
        for version_folder in os.listdir(dataset_path):
            params_path = os.path.join(dataset_path, version_folder, 'params.json')
            
            if os.path.isfile(params_path):
                with open(params_path, 'r') as file:
                    params = json.load(file)
                    
                    # Convert metric strings to numeric values
                    for key in params:
                        if isinstance(params[key], str) and ('+-' in params[key] or '±' in params[key]):
                            params[key] = extract_numeric_value(params[key])
                    
                    data.append(params)
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Convert all numeric columns to float
    numeric_columns = ['learning_rate', 'batch_size', 'negative_ratio', 'temperature',
                      'test_auc', 'test_pr_auc', 'test_f1', 'test_mcc',
                      
                      ]
    
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Create visualizations
    create_correlation_heatmap(df)
    create_performance_comparisons(df)
    create_hyperparameter_analysis(df)
    
    return df

def create_correlation_heatmap(df):
    """Create correlation heatmap for numeric columns"""
    # Select only numeric columns
    numeric_df = df.select_dtypes(include=[np.number])
    
    # Calculate correlation
    corr = numeric_df.corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
                annot_kws={'size': 8}, square=True, linewidths=0.5)
    plt.title("Correlation between Parameters and Metrics", fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png')
    plt.close()

def create_performance_comparisons(df):
    """Create visualizations comparing performance metrics"""
    metrics = ['test_auc', 'test_pr_auc', 'test_f1', 'test_mcc']
    
    for metric in metrics:
        if metric in df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(x='dataset_id', y=metric, data=df)
            plt.title(f"{metric} across Datasets")
            plt.xlabel("Dataset ID")
            plt.ylabel(metric)
            plt.tight_layout()
            plt.savefig(f'{metric}_comparison.png')
            plt.close()

def create_hyperparameter_analysis(df):
    """Analyze impact of hyperparameters on performance"""
    hyperparams = ['learning_rate', 'batch_size', 'negative_ratio', 'temperature']
    metrics = ['test_auc', 'test_pr_auc']
    
    for param in hyperparams:
        if param in df.columns:
            plt.figure(figsize=(12, 5))
            
            for i, metric in enumerate(metrics):
                if metric in df.columns:
                    plt.subplot(1, 2, i+1)
                    # Ensure both columns are numeric
                    plot_data = df[[param, metric]].dropna()
                    if not plot_data.empty:
                        sns.scatterplot(data=plot_data, x=param, y=metric)
                        plt.title(f"{param} vs {metric}")
            
            plt.tight_layout()
            plt.savefig(f'{param}_analysis.png')
            plt.close()

# Execute analysis
df = analyze_results()

# Print summary statistics
print("\nSummary Statistics:")
metrics = ['test_auc', 'test_pr_auc', 'test_f1', 'test_mcc']
print(df[metrics].describe())

# Find best configurations
print("\nBest Configurations:")
for metric in metrics:
    if metric in df.columns:
        best_idx = df[metric].idxmax()
        print(f"\nBest configuration for {metric}:")
        params_to_show = ['dataset_id', 'learning_rate', 'batch_size', 'negative_ratio', 'temperature']
        for param in params_to_show:
            if param in df.columns:
                print(f"{param}: {df.loc[best_idx, param]}")
        print(f"Score: {df.loc[best_idx, metric]}")

# Save processed data
df.to_csv('analyzed_results.csv', index=False)


Summary Statistics:
         test_auc  test_pr_auc     test_f1    test_mcc
count  720.000000   690.000000  690.000000  682.000000
mean     0.777837     0.168022    0.049341    0.005821
std      0.158011     0.120714    0.009452    0.003000
min      0.392200     0.019500    0.001776   -0.039100
25%      0.630000     0.042150    0.050700    0.005200
50%      0.835850     0.168150    0.050700    0.006200
75%      0.916475     0.271050    0.050800    0.007300
max      0.987250     0.459200    0.166100    0.011600

Best Configurations:

Best configuration for test_auc:
dataset_id: 1002
learning_rate: 0.001
batch_size: 32
negative_ratio: 50
temperature: 1.0
Score: 0.9872503884840264

Best configuration for test_pr_auc:
dataset_id: 1
learning_rate: 0.001
batch_size: 16
negative_ratio: 10
temperature: 1.0
Score: 0.4592

Best configuration for test_f1:
dataset_id: 1005
learning_rate: 0.001
batch_size: 32
negative_ratio: 50
temperature: 1.0
Score: 0.1661

Best configuration for test_mcc:
datase

In [4]:
!pwd

/scratch/ab9738/dfdl_imputation/contrastive_learning
